# scConcept Tutorial: Concept-level Interpretation of Single-Cell Topics

This tutorial demonstrates how to perform concept-level analysis using **scConcept**, 
given a trained topic model (e.g., ECRTM).

---

## Prerequisites

Before running this tutorial, you should have already:

1. Trained a topic model on your single-cell RNA-seq dataset using **ECRTM**.  
2. Saved the topic gene lists (e.g., top genes per topic) to disk.

---

## What this tutorial does

Starting from precomputed topic gene lists, this tutorial walks through:

1. **Loading topic gene lists** generated by ECRTM  
2. **Distilling topics into biologically meaningful concepts** using a large language model (LLM)  
3. **Assigning each cell to a concept** based on gene expression  
4. **Evaluating concept quality** using clustering metrics (Purity, AMI/NMI, ARI)  
5. **Assessing interpretability** using topic coherence (TC) and topic diversity (TD)  
6. **Visualizing concept assignments** alongside ground-truth cell types using UMAP  

---

## Expected outcome

By the end of this tutorial, you will obtain:

- A set of **interpretable biological concepts** derived from topic gene lists  
- **Cell-level concept annotations**  
- Quantitative evaluation of both **clustering performance** and **interpretability**  
- UMAP visualizations comparing **concepts with true cell types**

---

## Note

This tutorial assumes that the topic model training step has already been completed.  
If not, please first run the ECRTM pipeline to generate topic gene lists.

---

## Overview

This tutorial illustrates how **scConcept bridges neural topic models and biological interpretation**  
by transforming topic gene lists into **human-interpretable, concept-level representations**.

In [ ]:
import numpy as np
from types import SimpleNamespace
from pathlib import Path
import sys

# =========================================================
# Project setup for the tutorial notebook
# =========================================================

PROJECT_ROOT = Path.cwd().parent
ECRTM_PATH = PROJECT_ROOT / "ECRTM"
sys.path.append(str(ECRTM_PATH))

from singlecell_dataset import SingleCellDataset

# =========================================================
# User-defined configuration
# =========================================================

dataset_name = "pollen"
data_dir = PROJECT_ROOT / "Datasets"
batch_size = 512

# =========================================================
# Build configuration namespace
# =========================================================

config = SimpleNamespace(
    dataset=dataset_name,
    batch_size=batch_size,
    data_dir=str(data_dir),
)

# =========================================================
# Load dataset
# =========================================================

dataset = SingleCellDataset(
    dataset_name=config.dataset,
    batch_size=config.batch_size,
    data_dir=config.data_dir,
)

train_matrix = dataset.train_data
test_matrix = dataset.test_data
train_labels = dataset.train_labels
test_labels = dataset.test_labels
gene_names = dataset.gene_names
num_genes = dataset.n_genes

# =========================================================
# Sanity check
# =========================================================

print(f"Train data shape: {train_matrix.shape}")
print(f"Test data shape: {test_matrix.shape}")
print(f"Train labels shape: {np.shape(train_labels)}")
print(f"Test labels shape: {np.shape(test_labels)}")
print(f"Number of genes: {num_genes}")
print(f"First 10 genes: {gene_names[:10]}")

===> Train size: 301
===> Test size: 301
===> Number of genes: 10000
===> Avg expression per cell: 386106.977
===> Number of labels: 11
Train data shape: torch.Size([301, 10000])
Test data shape: torch.Size([301, 10000])
Train labels shape: (1, 301)
Test labels shape: (1, 301)
Number of genes: 10000
First 10 genes: ['A2LD1', 'A2M', 'A2M-AS1', 'A2ML1', 'A2MP1', 'A4GALT', 'AACS', 'AACSP1', 'AAMP', 'AANAT']


In [5]:
from pathlib import Path

# =========================================================
# Function: load topic gene lists
# =========================================================

def load_topic_genes(topic_file):
    """
    Load topic gene lists from a text file.

    Each line corresponds to one topic, where genes are separated by spaces.

    Parameters
    ----------
    topic_file : str or Path
        Path to the topic gene file.

    Returns
    -------
    list[list[str]]
        A list of topics, where each topic is a list of gene names.
    """
    topic_gene_lists = []

    with open(topic_file, "r", encoding="utf-8") as f:
        for line in f:
            genes = line.strip().split()
            topic_gene_lists.append(genes)

    return topic_gene_lists


# =========================================================
# Automatically construct topic file path
# =========================================================

num_topics = 50
seed = 1
epoch = 500

topic_dir = (
    PROJECT_ROOT
    / "ECRTM"
    / "output"
    / "topics"
    / f"{dataset_name}_K{num_topics}_seed{seed}"
)

topic_file = topic_dir / f"epoch{epoch}_top_genes.txt"

# Check file existence
if not topic_file.exists():
    raise FileNotFoundError(f"Topic file not found: {topic_file}")

# =========================================================
# Load topic gene lists
# =========================================================

topic_gene_lists = load_topic_genes(topic_file)

# =========================================================
# Sanity check
# =========================================================

print(f"Topic file: {topic_file}")
print(f"Number of topics: {len(topic_gene_lists)}")
print(f"Top 20 genes in Topic 0: {topic_gene_lists[0][:20]}")

Topic file: /home/mcb/users/hchen26/method/textgrad/LLM-ITL-main/scConcept_github/ECRTM/output/topics/pollen_K50_seed1/epoch500_top_genes.txt
Number of topics: 50
Top 20 genes in Topic 0: ['RUNX3', 'BZRAP1', 'RASSF2', 'DOK3', 'IGSF10', 'TARBP1', 'PREX1', 'TRIM13', 'USF2', 'PVRL1', 'ECM2', 'RECQL4', 'WDR52', 'PDGFD', 'SPAG8', 'DISC2', 'AP4B1', 'SLC25A15', 'ANKRD36BP2', 'ATG16L2']


In [ ]:
import json
from pathlib import Path
from openai import OpenAI

# =========================================================
# Convert topic gene lists into JSON payload
# =========================================================
# Each topic is represented by:
# - topic_id: topic index
# - genes: top genes of that topic

topics_payload = [
    {
        "topic_id": f"topic_{topic_id}",
        "genes": topic_gene_lists[topic_id]
    }
    for topic_id in range(len(topic_gene_lists))
]

topics_json_str = json.dumps({"topics": topics_payload}, indent=2)

# =========================================================
# Build the topic distillation prompt
# =========================================================
# The LLM is asked to merge similar topics into biologically
# coherent concepts based only on the top gene lists.

topic_distillation_prompt = f"""
You are given topics derived from a neural topic model on single-cell RNA-seq.

Each topic includes ONLY:
- A list of the top 100 genes for that topic,
  sorted from highest weight to lowest weight in the model.

Here are all topics (JSON list):
{topics_json_str}

TASK:
You must perform biological topic distillation by analyzing ONLY the top-100 gene lists.

Specifically:
1. Identify topics whose top genes indicate highly similar biological pathways or cell states.
2. Merge such topics into unified, biologically coherent concepts.
3. Remove topics that are:
   - too vague (gene list does not form a coherent module),
   - too narrow (dominated by a single outlier gene),
   - biologically uninterpretable given the provided gene patterns.

For each FINAL biological concept, output:
- "name": a concise 2-4 word biological label.
      * Avoid generic labels like "General Regulation".
      * Prefer pathway- or state-specific terms.
- "description": a 10-30 word natural language description
      * Summarize the biological meaning of this concept.
      * Describe the core pathway, cellular program, or functional state.
      * Must be consistent with the concept name and gene list.
      * Do NOT mention topic IDs, gene counts, or modeling details.
- "genes": EXACTLY 100 representative genes.
      * MUST be chosen from the union of gene lists of the merged source topics.
      * MUST reflect a coherent pathway or co-expression program.
      * MUST respect the original sorted importance: higher-ranked genes are preferred.
      * Do NOT invent gene names.
- "source_topics": list of topic IDs merged into this concept.

OUTPUT FORMAT (STRICT):
Respond ONLY with a valid JSON object in EXACTLY this form:

{{
  "concepts": [
    {{
      "name": "Example Name",
      "description": "A concise biological description summarizing the core pathway or cellular program represented by this concept.",
      "genes": ["GENE1", "GENE2", "GENE3", "...", "GENE100"],
      "source_topics": ["topic_0", "topic_3"]
    }}
  ]
}}

ABSOLUTE RULES:
- DO NOT output anything outside the JSON.
- DO NOT add code fences.
- DO NOT invent genes.
- The final genes must reflect the most informative subset of the top 100 genes per topic.
"""

# =========================================================
# Call the OpenAI API to generate concepts
# =========================================================
# Make sure your OPENAI_API_KEY environment variable is set
# before running this step.

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-5",
    seed=1,
    messages=[
        {"role": "system", "content": "You are an expert in single-cell biology."},
        {"role": "user", "content": topic_distillation_prompt}
    ]
)

response_text = completion.choices[0].message.content

# =========================================================
# Parse the LLM output
# =========================================================
# The model is expected to return a JSON object with a
# top-level key called "concepts".

concept_result = json.loads(response_text)
concepts = concept_result["concepts"]

print(f"Number of concepts generated: {len(concepts)}")
print(f"First concept name: {concepts[0]['name']}")

# =========================================================
# Save the generated concepts
# =========================================================
# Concepts are saved to:
# Results/<dataset_name>/gpt5_<dataset_name>_concepts.json

results_dir = PROJECT_ROOT / "Results" / dataset_name
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / f"gpt5_{dataset_name}_concepts.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(concepts, f, indent=2, ensure_ascii=False)

print(f"Saved concepts to: {output_file}")

In [ ]:
import numpy as np
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# =========================================================
# Utility functions for clustering evaluation
# =========================================================

def compute_clustering_metrics(true_labels, score_matrix):
    """
    Compute purity, adjusted mutual information (AMI), and adjusted Rand index (ARI)
    based on the highest-scoring concept assigned to each cell.

    Parameters
    ----------
    true_labels : array-like
        Ground-truth labels for cells.
    score_matrix : np.ndarray
        Matrix of shape (num_cells, num_concepts), where each row contains
        concept scores for one cell.

    Returns
    -------
    purity : float
        Clustering purity.
    ami : float
        Adjusted mutual information.
    ari : float
        Adjusted Rand index.
    """
    true_clusters = {}
    predicted_clusters = {}
    num_cells = len(true_labels)

    predicted_indices = np.argmax(score_matrix, axis=1)

    # Group cells by ground-truth labels
    for cell_index, label in enumerate(true_labels):
        true_clusters.setdefault(label, set()).add(cell_index)

    # Group cells by predicted concept assignment
    for cell_index, concept_index in enumerate(predicted_indices):
        cluster_name = f"Concept_{concept_index}"
        predicted_clusters.setdefault(cluster_name, set()).add(cell_index)

    # Compute purity
    correct_count = 0
    for predicted_cells in predicted_clusters.values():
        best_overlap = 0
        for true_cells in true_clusters.values():
            best_overlap = max(best_overlap, len(predicted_cells.intersection(true_cells)))
        correct_count += best_overlap

    purity = correct_count / num_cells
    ami = adjusted_mutual_info_score(true_labels, predicted_indices)
    ari = adjusted_rand_score(true_labels, predicted_indices)

    return purity, ami, ari


def flatten_labels(labels):
    """
    Convert labels into a one-dimensional NumPy array.

    This is useful when labels are stored as nested arrays or object arrays.

    Parameters
    ----------
    labels : array-like
        Input labels.

    Returns
    -------
    np.ndarray
        Flattened label array.
    """
    labels = np.array(labels, dtype=object).reshape(-1)
    flattened = []

    for label in labels:
        if isinstance(label, np.ndarray):
            if label.size == 1:
                flattened.append(label.item())
            else:
                flattened.append(tuple(label.tolist()))
        else:
            flattened.append(label)

    return np.array(flattened)


# =========================================================
# Utility functions for concept-based cell annotation
# =========================================================

def build_gene_index(gene_names):
    """
    Create a mapping from gene name to column index.

    Parameters
    ----------
    gene_names : list-like
        Ordered list of gene names.

    Returns
    -------
    dict
        Dictionary mapping gene name to column index.
    """
    return {str(gene): idx for idx, gene in enumerate(gene_names)}


def truncate_concept_genes(concepts, top_k):
    """
    Keep only the top-k genes for each concept.

    Parameters
    ----------
    concepts : list[dict]
        List of concept dictionaries. Each concept should contain at least:
        - "name"
        - "genes"
        - optional "source_topics"
    top_k : int
        Number of top genes to retain.

    Returns
    -------
    list[dict]
        Updated concept list with truncated gene sets.
    """
    truncated_concepts = []

    for concept in concepts:
        truncated_concepts.append(
            {
                "name": concept["name"],
                "genes": concept["genes"][:top_k],
                "source_topics": concept.get("source_topics", [])
            }
        )

    return truncated_concepts


def assign_cells_to_concepts_by_zscore(concepts, expression_matrix, gene_to_index):
    """
    Assign each cell to the concept with the highest average marker z-score.

    Parameters
    ----------
    concepts : list[dict]
        List of concept dictionaries. Each concept should contain:
        - "name"
        - "genes"
    expression_matrix : torch.Tensor
        Cell-by-gene expression matrix of shape (num_cells, num_genes).
    gene_to_index : dict
        Mapping from gene name to column index.

    Returns
    -------
    score_matrix : np.ndarray
        Matrix of shape (num_cells, num_concepts).
    concept_names : list[str]
        Names of all concepts.
    predicted_labels : list[str]
        Predicted concept label for each cell.
    """
    expression_array = expression_matrix.detach().cpu().numpy().astype(float)
    num_cells, _ = expression_array.shape

    # Standardize each gene across cells
    gene_mean = expression_array.mean(axis=0, keepdims=True)
    gene_std = expression_array.std(axis=0, keepdims=True) + 1e-6
    expression_zscore = (expression_array - gene_mean) / gene_std

    concept_names = [concept["name"] for concept in concepts]
    num_concepts = len(concepts)

    score_matrix = np.zeros((num_cells, num_concepts), dtype=float)

    for concept_index, concept in enumerate(concepts):
        concept_genes = concept["genes"]
        gene_indices = [
            gene_to_index[gene] for gene in concept_genes if gene in gene_to_index
        ]

        if len(gene_indices) == 0:
            continue

        score_matrix[:, concept_index] = expression_zscore[:, gene_indices].mean(axis=1)

    predicted_indices = np.argmax(score_matrix, axis=1)
    predicted_labels = [concept_names[idx] for idx in predicted_indices]

    return score_matrix, concept_names, predicted_labels


# =========================================================
# Example usage: concept-based annotation on the test set
# =========================================================

gene_to_index = build_gene_index(gene_names)

top_k_concepts = truncate_concept_genes(concepts, top_k=100)

score_matrix, concept_names, predicted_concepts = assign_cells_to_concepts_by_zscore(
    top_k_concepts,
    test_matrix,
    gene_to_index
)

print(f"Score matrix shape: {score_matrix.shape}")
print(f"Example predicted concepts: {predicted_concepts[:10]}")


# =========================================================
# Evaluate concept-based assignments
# =========================================================

true_test_labels = flatten_labels(test_labels)

print(f"Label example type: {type(true_test_labels[0])}")
print(f"First 10 ground-truth labels: {true_test_labels[:10]}")

purity, ami, ari = compute_clustering_metrics(true_test_labels, score_matrix)

print(f"Purity: {purity:.4f}")
print(f"ARI: {ari:.4f}")
print(f"AMI: {ami:.4f}")

Scores shape: (301, 8)
Example predicted concepts: ['Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium', 'Inflamed luminal epithelium']
<class 'numpy.str_'> ['2338' '2338' '2338' '2338' '2338' '2338' '2338' '2338' '2338' '2338']
Purity: 0.840531561461794
ARI: 0.8527256060437747
NMI: 0.9038008254415674


In [ ]:
import numpy as np
from collections import Counter
from pathlib import Path
import gseapy as gp

# =========================================================
# Utility function: compute topic coherence (TC)
# =========================================================

def compute_topic_coherence_from_gene_lists(
    background_matrix,
    background_vocab,
    concept_gene_lists,
    gene_index_mapping,
    top_n=10
):
    """
    Compute topic coherence (TC) from concept gene lists.

    Parameters
    ----------
    background_matrix : np.ndarray
        Binary pathway-by-gene matrix of shape (num_pathways, num_background_genes).
    background_vocab : list[str]
        Background gene vocabulary corresponding to the columns of background_matrix.
    concept_gene_lists : list[list[str]]
        Concept gene lists, where each concept is represented by a list of gene names.
    gene_index_mapping : dict
        Mapping from vocabulary index to background_matrix column index.
    top_n : int, default=10
        Number of top genes used for coherence calculation.

    Returns
    -------
    float
        Average topic coherence score across all concepts.
    """
    num_concepts = len(concept_gene_lists)
    num_documents = background_matrix.shape[0]

    gene_to_vocab_index = {gene: idx for idx, gene in enumerate(background_vocab)}

    # Convert gene lists into index lists
    indexed_concepts = []
    for genes in concept_gene_lists:
        gene_indices = []

        for gene in genes:
            gene_indices.append(gene_to_vocab_index.get(gene, -1))

        if len(gene_indices) > top_n:
            gene_indices = gene_indices[:top_n]
        elif len(gene_indices) < top_n:
            gene_indices = gene_indices + [-1] * (top_n - len(gene_indices))

        indexed_concepts.append(np.array(gene_indices, dtype=int))

    total_coherence = 0.0

    for concept_indices in indexed_concepts:
        concept_score = 0.0

        for i in range(top_n):
            gene_i = concept_indices[i]

            if gene_i in gene_index_mapping:
                presence_i = background_matrix[:, gene_index_mapping[gene_i]] > 0
                prob_i = np.sum(presence_i) / num_documents

                for j in range(i + 1, top_n):
                    gene_j = concept_indices[j]

                    if gene_j in gene_index_mapping:
                        presence_j = background_matrix[:, gene_index_mapping[gene_j]] > 0
                        count_j = np.sum(presence_j)
                        count_ij = np.sum(presence_i * presence_j)

                        if prob_i * count_j * count_ij > 0:
                            prob_j = count_j / num_documents
                            prob_ij = count_ij / num_documents

                            if (prob_i * prob_j) == 1 and prob_ij == 1:
                                continue

                            concept_score += np.log(prob_ij / (prob_i * prob_j)) / -np.log(prob_ij)

        total_coherence += concept_score * (2 / (top_n * top_n - top_n))

    return total_coherence / num_concepts


# =========================================================
# Utility function: compute topic diversity (TD)
# =========================================================

def compute_topic_diversity(concept_gene_lists):
    """
    Compute topic diversity (TD), defined as the fraction of genes
    that appear exactly once across all concept gene lists.

    Parameters
    ----------
    concept_gene_lists : list[list[str]]
        List of concept gene lists.

    Returns
    -------
    float
        Topic diversity score.
    """
    num_concepts = len(concept_gene_lists)
    genes_per_concept = len(concept_gene_lists[0])

    all_genes = []
    for gene_list in concept_gene_lists:
        all_genes.extend(gene_list)

    gene_counts = Counter(all_genes)
    unique_gene_count = sum(1 for count in gene_counts.values() if count == 1)

    return unique_gene_count / (num_concepts * genes_per_concept)


# =========================================================
# Load pathway gene sets from GMT
# =========================================================
# This example uses the MSigDB KEGG collection as the background
# reference for concept coherence evaluation.

gmt_file = PROJECT_ROOT / "Datasets" / "msigdb.v2024.1.Hs.symbols.gmt"

if not gmt_file.exists():
    raise FileNotFoundError(f"GMT file not found: {gmt_file}")

kegg_pathways = gp.read_gmt(path=str(gmt_file))

# =========================================================
# Build the background vocabulary
# =========================================================

background_vocab = sorted(
    list({gene.upper() for genes in kegg_pathways.values() for gene in genes})
)
background_gene_to_index = {
    gene: idx for idx, gene in enumerate(background_vocab)
}

# Build a binary pathway-by-gene matrix
background_matrix = np.zeros(
    (len(kegg_pathways), len(background_vocab)),
    dtype=float
)

for pathway_index, genes in enumerate(kegg_pathways.values()):
    for gene in genes:
        background_matrix[pathway_index, background_gene_to_index[gene.upper()]] = 1.0

# =========================================================
# Prepare concept gene lists
# =========================================================
# Convert concept genes to uppercase and keep the top 10 genes
# for each concept when evaluating TC and TD.

concept_gene_lists_upper = [
    [str(gene).upper() for gene in concept.get("genes", [])[:10]]
    for concept in top_k_concepts
]

# =========================================================
# Build the vocabulary-to-column mapping
# =========================================================
# Here the background vocabulary and matrix columns are aligned,
# so the mapping is simply identity.

gene_index_mapping = {idx: idx for idx in range(len(background_vocab))}

# =========================================================
# Compute TC and TD
# =========================================================

topic_coherence = compute_topic_coherence_from_gene_lists(
    background_matrix=background_matrix,
    background_vocab=background_vocab,
    concept_gene_lists=concept_gene_lists_upper,
    gene_index_mapping=gene_index_mapping,
    top_n=10
)

topic_diversity = compute_topic_diversity(concept_gene_lists_upper)

print(f"TC: {topic_coherence:.4f}")
print(f"TD: {topic_diversity:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

# =========================================================
# Prepare the expression matrix and labels
# =========================================================

expression_array = test_matrix.detach().cpu().numpy()
true_cell_types = flatten_labels(test_labels)
predicted_concepts_array = np.array(predicted_concepts)

print(f"Expression matrix shape: {expression_array.shape}")
print(f"Cell type labels shape: {true_cell_types.shape}")
print(f"Predicted concept labels shape: {predicted_concepts_array.shape}")

# =========================================================
# Build an AnnData object for visualization
# =========================================================

adata_visualization = ad.AnnData(expression_array)

adata_visualization.obs["cell_type"] = pd.Categorical(true_cell_types)
adata_visualization.obs["concept"] = pd.Categorical(predicted_concepts_array)

# Add gene names
adata_visualization.var_names = [str(gene) for gene in gene_names]

# =========================================================
# Compute PCA, neighborhood graph, and UMAP
# =========================================================

sc.pp.pca(adata_visualization, n_comps=50)
sc.pp.neighbors(adata_visualization, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata_visualization)

# =========================================================
# Visualize UMAP colored by cell type and concept
# =========================================================

figure = sc.pl.umap(
    adata_visualization,
    color=["cell_type", "concept"],
    ncols=2,
    frameon=False,
    wspace=0.35,
    legend_loc="right margin",
    return_fig=True
)

# Optionally enlarge legend font size
for axis in figure.axes[:2]:
    if axis.legend_ is not None:
        for text in axis.legend_.get_texts():
            text.set_fontsize(12)

# =========================================================
# Save the figure
# =========================================================

results_dir = PROJECT_ROOT / "Results" / dataset_name
results_dir.mkdir(parents=True, exist_ok=True)

figure_file = results_dir / "umap_celltype_concept.pdf"
figure.savefig(figure_file, bbox_inches="tight")

print(f"UMAP figure saved to: {figure_file}")